# World Bank country-level context exercise

**Programme:** Operational Data Analytics and Dashboard Engineering for SupTech Systems
**Day 1, separate exercise.** This notebook is deliberately separate from the
ingestion pipeline.

> The indicators in this notebook are real, published World Bank figures. They are
> country-level. They are never joined to the institution-level synthetic
> submissions, and the notebook explains why.

**The point of this exercise.** Supervisory reporting and public statistics look
like the same kind of data and are not. One row of a country indicator is a
country's value in a year. One row of an institution submission is one
institution's position on one date. Joining them would claim that a country
figure is an institution fact.

**Learning outcomes**

1. Retrieve a published indicator and read its metadata.
2. State the grain of a country-level observation.
3. Explain why country-level and institution-level data must not be joined.
4. Recognise a gap in a published series, and why a gap is not a zero.

## 1. Retrieve an indicator

The World Bank API returns two elements. The first is metadata about the response.
The second is the array of observations.

Always read the metadata block first. It tells you whether you are looking at the
whole series or one page of it, and it tells you when the source was last updated.

In [ ]:
import json
import urllib.request

API_ROOT = "https://api.worldbank.org/v2/"

#: ISO 3166-1 alpha-3 code for Tanzania.
COUNTRY = "TZA"

#: Account ownership at a financial institution or with a mobile money service
#: provider, as a percentage of the population aged 15 and over.
INDICATOR = "FX.OWN.TOTL.ZS"


def fetch(url):
    """GET a JSON document from the World Bank API."""
    with urllib.request.urlopen(url, timeout=60) as response:
        return json.loads(response.read().decode("utf-8"))


def indicator_url(country, indicator, per_page=100, source=None):
    url = (
        f"{API_ROOT}country/{country}/indicator/{indicator}"
        f"?format=json&per_page={per_page}"
    )
    if source:
        # Requesting more than one indicator in a single call requires the
        # source parameter, otherwise the API rejects the call.
        url += f"&source={source}"
    return url


metadata, observations = fetch(indicator_url(COUNTRY, INDICATOR))

print("Response metadata")
for key in ("page", "pages", "per_page", "total", "sourceid", "lastupdated"):
    print(f"  {key}: {metadata.get(key)}")

print()
print("Observations:", len(observations))
if observations:
    first = observations[0]
    print("Indicator:", first["indicator"]["value"])
    print("Unit:", first.get("unit"))
    print("Country:", first["country"]["value"])

## 2. The grain of a country-level observation

One row is one country, one indicator, one year. The years that are missing from a
published series are missing because the source did not publish an observation for
that year. They are not zeros, and filling them with zeros would be the same error
as turning a missing deposit balance into a zero.

In [ ]:
import pandas as pd

country_frame = pd.DataFrame(
    [
        {
            "country_code": row["countryiso3code"],
            "country": row["country"]["value"],
            "indicator_code": row["indicator"]["id"],
            "indicator": row["indicator"]["value"],
            "year": int(row["date"]),
            "value": row["value"],
        }
        for row in observations
    ]
).sort_values("year", ascending=False)

print("Rows:", len(country_frame))
print("Distinct countries:", country_frame["country_code"].nunique())
print("Distinct indicators:", country_frame["indicator_code"].nunique())
print("Years covered:", country_frame["year"].min(), "to", country_frame["year"].max())
print("Missing values:", int(country_frame["value"].isna().sum()))

print()
print("Most recent observations:")
print(country_frame.head(10).to_string(index=False))

### A gap is not a zero

Look at the `value` column above. Where the source published nothing, the cell is
empty. If you filled those cells with zero, you would be asserting that account
ownership in Tanzania was zero percent in that year. That is a claim about the
world, and the source does not support it.

This is the same discipline as the missing deposit balance in the ingestion
pipeline. A missing value is information about what the supervisor does not know.
Replacing it with a number destroys the information and invents a fact.

## 3. Several indicators at once

The API can return more than one indicator in a single call, separated by a
semicolon. That call needs the `source` parameter, and without it the API rejects
the request. This is the shape a later cross-country or multi-indicator view takes.

In [ ]:
SECOND_INDICATOR = "FS.AST.PRVT.GD.ZS"

metadata_two, observations_two = fetch(
    indicator_url(COUNTRY, f"{INDICATOR};{SECOND_INDICATOR}", source=2)
)

print("Pages:", metadata_two.get("pages"), "Total:", metadata_two.get("total"))

multi_frame = pd.DataFrame(
    [
        {
            "indicator_code": row["indicator"]["id"],
            "indicator": row["indicator"]["value"],
            "year": int(row["date"]),
            "value": row["value"],
        }
        for row in observations_two
    ]
)

pivot = multi_frame.pivot_table(
    index="year", columns="indicator_code", values="value", dropna=False
).sort_index(ascending=False)

print(pivot.head(8).to_string())

## 4. The two grains, side by side

This is the part worth doing slowly. The institution-level submissions and the
country-level indicators are both "financial data about Tanzania". They cannot be
treated as the same kind of observation.

In [ ]:
# The institution-level grain, taken from the Day 1 synthetic submissions.
submission_grain = pd.DataFrame(
    [
        {
            "source": "institution submission",
            "one row is": "one institution, one reporting period",
            "identified by": "institution_id + reporting_period",
            "frequency": "monthly, per institution",
            "produced by": "a supervised institution, for the supervisor",
        },
        {
            "source": "World Bank indicator",
            "one row is": "one country, one indicator, one year",
            "identified by": "country code + indicator code + year",
            "frequency": "annual, per country",
            "produced by": "a national statistic, published for everyone",
        },
    ]
)

print(submission_grain.to_string(index=False))

### Why the join does not work

Suppose the two are joined on country. Every institution in Tanzania would receive
the country's account ownership figure.

- The country figure would be repeated across every institution row, which dilutes
  it and makes it look like an institution-level observation.
- The institution figure would be contaminated by a national statistic that says
  nothing about that institution.
- An analyst computing a sector aggregate would silently double-count the country
  value once per institution.

The result would look reasonable in a table and be meaningless. That is what makes
this the most dangerous mistake in a SupTech project, and why the grains are
documented separately in the data model rather than left to whoever writes the join.

## 5. Comparison countries

A second, safe use of country indicators: comparing countries. This is a
country-to-country comparison, so the grain is consistent throughout and there is no
join to institution data.

In [ ]:
COMPARISON_COUNTRIES = {
    "TZA": "Tanzania",
    "KEN": "Kenya",
    "UGA": "Uganda",
    "RWA": "Rwanda",
    "ZMB": "Zambia",
    "ZWE": "Zimbabwe",
}

rows = []
for code_, label in COMPARISON_COUNTRIES.items():
    try:
        _, country_observations = fetch(indicator_url(code_, INDICATOR))
    except Exception as error:  # noqa: BLE001 - the point is to show the gap
        print(f"{label}: retrieval failed ({error})")
        continue
    for observation in country_observations:
        if observation["value"] is not None:
            rows.append(
                {
                    "country_code": code_,
                    "country": label,
                    "year": int(observation["date"]),
                    "value": observation["value"],
                }
            )

comparison = pd.DataFrame(rows)
latest = comparison.sort_values("year").groupby("country_code").tail(1)
print("Most recent published observation per country:")
print(latest.sort_values("value", ascending=False).to_string(index=False))

## 6. What to take away

1. One row of a country indicator is a country, an indicator and a year. That is a
   different observation from an institution submission.
2. The metadata block is part of the data. It carries the page count and the last
   updated date, and a figure without a date is not a figure.
3. A gap in a published series is missing information, not a zero.
4. Country indicators are used as context and as country-to-country comparison.
   They are never joined to institution-level submissions.
5. Before teaching or quoting any indicator, read its definition, its unit, its
   coverage and its update frequency from the source metadata.

**Sources**

- World Bank API root: https://api.worldbank.org/v2/
- API documentation: https://datahelpdesk.worldbank.org/knowledgebase/articles/889392
- Financial Development data collection: https://data.worldbank.org/collections/findata